# 3D Scan Pipeline - Part 2: Training

**Objective:** Train the Gaussian Splatting model using Taichi and Export to .splat.
**Input:** Upload `3d_scan_data_part1.zip` from Part 1 to the notebook environment.

**Environment:** **GPU REQUIRED (T4 or better)**. 
**Note:** This version attempts to use the default Kaggle/Colab NumPy environment (likely 2.0+). If you encounter weird "symbol not found" errors, we may need to revert to strict versioning.

In [ ]:
import os
import sys

print("⏳ Setting up Environment (Part 2)...")

# 1. Clone Repo (Only if local project not found)
if not os.path.exists("3DSCAN"):
    !git clone https://github.com/PRIDA-TAKON/3DSCAN.git
    if os.path.exists("3DSCAN"):
        os.chdir("3DSCAN")
else:
    print("📂 Project folder found. Using local version.")
    if os.path.basename(os.getcwd()) != "3DSCAN" and os.path.exists("3DSCAN"):
        os.chdir("3DSCAN")

# 2. Install Dependencies
print("⏳ Installing Dependencies...")
!pip install --upgrade pip
# Removed strict numpy<2.0 to try using Kaggle's default
!pip install --upgrade numba scipy pandas scikit-learn opencv-python opencv-python-headless opencv-contrib-python matplotlib pillow plyfile tqdm roma
!pip install taichi

# Install Taichi Splatting (Wanmeihuali Version)
if os.path.exists("taichi_3d_gaussian_splatting"):
    print("📂 taichi_3d_gaussian_splatting found. Skipping clone.")
else:
    !git clone --depth 1 https://github.com/wanmeihuali/taichi_3d_gaussian_splatting.git

# Standard install without forcing numpy version
!pip install -r taichi_3d_gaussian_splatting/requirements.txt
!pip install ./taichi_3d_gaussian_splatting

# Verify Environment
try:
    import numpy
    print(f"✅ Setup Complete. NumPy Version: {numpy.__version__}")
    if int(numpy.__version__.split('.')[0]) >= 2:
        print("ℹ️ Note: Running with NumPy 2.0+. If this fails, we will need to downgrade.")
except Exception as e:
    print(f"⚠️ Error checking numpy: {e}")

In [ ]:
print("=== Import Data from Part 1 ===")
import glob
import zipfile

# Look for the zip file (uploaded by user)
zip_candidates = glob.glob("**/*.zip", recursive=True)
zip_path = None

for p in zip_candidates:
    if "3d_scan_data_part1" in p or "3d_scan_output" in p: 
        zip_path = p
        break

if not zip_path:
    if os.path.exists("working_data/3d_scan/sparse"):
        print("📂 Found 'working_data/3d_scan' directory directly. Proceeding.")
    else:
        print("❌ No data zip found! Please upload '3d_scan_data_part1.zip' from Part 1.")
else:
    print(f"📦 Found data zip: {zip_path}")
    print("⏳ Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("✅ Extraction Complete.")

In [ ]:
print("=== STEP 3: Train Taichi Splatting ===")
if not os.path.exists("working_data/3d_scan"):
    print("❌ working_data/3d_scan not found. Check zip file structure.")
else:
    !python scripts/step3_train_splatting.py --project_path "working_data/3d_scan" --output_path "outputs/3d_scan/taichi_splatting"

In [ ]:
print("=== STEP 4: Export ===")
!python scripts/step4_export.py --input_parquet "outputs/3d_scan/taichi_splatting/model.parquet" --output_splat "outputs/3d_scan/taichi_splatting/model.splat"

In [ ]:
print("=== Compress Final Model for Download ===")
output_model_zip = "3d_splat_model.zip"

if os.path.exists("outputs/3d_scan/taichi_splatting"):
    !zip -r {output_model_zip} outputs/3d_scan/taichi_splatting
    
    from IPython.display import FileLink
    display(FileLink(output_model_zip))
else:
    print("❌ No output model found.")